# ARGUS — Notebook 1 v2: BDD100K Fine-Tune (P100 Fixed)

**Base:** `yolov8l.pt` (COCO pretrained)  
**Fine-tune on:** BDD100K subset — 20K dashcam images, 4 vehicle classes  
**Target:** 80–85% mAP50  
**Est. time:** ~4–5 hrs on Kaggle P100  

### Before running
1. Settings → Accelerator → **GPU P100**
2. Settings → Internet → **On** (needed to download yolov8l.pt)
3. Add dataset input: `a7madmostafa/bdd100k-yolo`
4. **Save Version → Save & Run All (Commit)**

In [1]:
                                                                                                       
  import subprocess                                                                                                                                                                                       
                                                                                                                                                                                                          
  print('Installing ultralytics...')                                                                                                                                                                      
  subprocess.run(['pip', 'install', '-q', 'ultralytics'], check=True)                                                                                                                                     
                                                            
  import torch                                                                                                                                                                                            
  print(f'Torch version  : {torch.__version__}')
  print(f'CUDA available : {torch.cuda.is_available()}')                                                                                                                                                  
  assert torch.cuda.is_available(), 'GPU not available'     
  print(f'GPU            : {torch.cuda.get_device_name(0)}')                                                                                                                                              
  print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
                                                                                                                            

Installing ultralytics...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.4 MB/s eta 0:00:00
Torch version  : 2.10.0+cu128
CUDA available : True
GPU            : Tesla T4
VRAM           : 15.6 GB


In [2]:
# ── Cell 2: Verify BDD100K dataset paths ──────────────────────────────────────
from pathlib import Path

BDD_ROOT   = Path('/kaggle/input/datasets/a7madmostafa/bdd100k-yolo')
TRAIN_IMGS = BDD_ROOT / 'train' / 'images'
TRAIN_LBLS = BDD_ROOT / 'train' / 'labels'
VAL_IMGS   = BDD_ROOT / 'val'   / 'images'
VAL_LBLS   = BDD_ROOT / 'val'   / 'labels'

for name, path in [('BDD root', BDD_ROOT), ('Train images', TRAIN_IMGS),
                   ('Train labels', TRAIN_LBLS), ('Val images', VAL_IMGS),
                   ('Val labels', VAL_LBLS)]:
    status = '✓' if path.exists() else '✗ MISSING'
    print(f'{name:15s}: {status}  ({path})')

assert BDD_ROOT.exists(),   f'BDD root not found: {BDD_ROOT}'
assert TRAIN_IMGS.exists(), f'Train images not found: {TRAIN_IMGS}'
assert TRAIN_LBLS.exists(), f'Train labels not found: {TRAIN_LBLS}'
assert VAL_IMGS.exists(),   f'Val images not found: {VAL_IMGS}'
assert VAL_LBLS.exists(),   f'Val labels not found: {VAL_LBLS}'

# Count files (fast — no rglob)
n_train = sum(1 for _ in TRAIN_IMGS.iterdir())
n_val   = sum(1 for _ in VAL_IMGS.iterdir())
print(f'\nTrain images: {n_train:,}')
print(f'Val images  : {n_val:,}')

BDD root       : ✓  (/kaggle/input/datasets/a7madmostafa/bdd100k-yolo)
Train images   : ✓  (/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/train/images)
Train labels   : ✓  (/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/train/labels)
Val images     : ✓  (/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/val/images)
Val labels     : ✓  (/kaggle/input/datasets/a7madmostafa/bdd100k-yolo/val/labels)

Train images: 70,000
Val images  : 10,000


In [3]:
# ── Cell 3: Detect BDD100K class mapping ──────────────────────────────────────
# BDD100K YOLO format can vary by dataset preparer.
# Auto-detect by sampling 200 label files.
import random
from pathlib import Path
from collections import Counter

random.seed(42)
sample_labels = random.sample(list(TRAIN_LBLS.glob('*.txt')), min(200, sum(1 for _ in TRAIN_LBLS.glob('*.txt'))))

class_counts = Counter()
for lf in sample_labels:
    for line in lf.read_text().strip().splitlines():
        if line.strip():
            class_counts[int(line.split()[0])] += 1

max_class = max(class_counts.keys())
print(f'Class IDs found: {sorted(class_counts.keys())}')
print(f'Max class ID   : {max_class}')

# Mapping A (most common for a7madmostafa):
# 0=person, 1=rider, 2=car, 3=truck, 4=bus, 5=train, 6=motorcycle, 7=bicycle, 8=traffic light, 9=traffic sign
#
# Mapping B (compact):
# 0=car, 1=truck, 2=bus, 3=motorcycle

if max_class >= 6:
    BDD_TO_ARGUS = {2: 0, 6: 1, 4: 2, 3: 3}  # car, motorcycle, bus, truck
    print('Detected: Mapping A (10-class)')
else:
    BDD_TO_ARGUS = {0: 0, 3: 1, 2: 2, 1: 3}
    print('Detected: Mapping B (compact)')

ARGUS_NAMES = ['car', 'motorcycle', 'bus', 'truck']
print(f'BDD→ARGUS map  : {BDD_TO_ARGUS}')
print('Classes        :', {k: ARGUS_NAMES[v] for k, v in BDD_TO_ARGUS.items()})

Class IDs found: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Max class ID   : 8
Detected: Mapping A (10-class)
BDD→ARGUS map  : {2: 0, 6: 1, 4: 2, 3: 3}
Classes        : {2: 'car', 6: 'motorcycle', 4: 'bus', 3: 'truck'}


In [4]:
# ── Cell 4: Build 20K subset with remapped labels ─────────────────────────────
import shutil
import random
from pathlib import Path
from collections import Counter

DATASET  = Path('/kaggle/working/dataset_argus')
SUBSET_N = 20000
VAL_N    = 2000
random.seed(42)

def build_split(src_imgs, src_lbls, split, max_n):
    dst_img = DATASET / split / 'images'
    dst_lbl = DATASET / split / 'labels'
    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    all_imgs = list(src_imgs.glob('*.jpg')) + list(src_imgs.glob('*.png'))
    random.shuffle(all_imgs)

    kept = 0
    skipped_no_label = 0
    skipped_no_vehicle = 0
    stats = Counter()

    for img_path in all_imgs:
        if kept >= max_n:
            break
        lbl_path = src_lbls / (img_path.stem + '.txt')
        if not lbl_path.exists():
            skipped_no_label += 1
            continue

        new_lines = []
        for line in lbl_path.read_text().strip().splitlines():
            parts = line.split()
            if not parts:
                continue
            cls = int(parts[0])
            if cls in BDD_TO_ARGUS:
                new_cls = BDD_TO_ARGUS[cls]
                new_lines.append(f'{new_cls} {" ".join(parts[1:])}')
                stats[ARGUS_NAMES[new_cls]] += 1

        if not new_lines:
            skipped_no_vehicle += 1
            continue

        shutil.copy(img_path, dst_img / img_path.name)
        (dst_lbl / (img_path.stem + '.txt')).write_text('\n'.join(new_lines))
        kept += 1

    print(f'{split}: {kept} images kept | no_label={skipped_no_label} | no_vehicle={skipped_no_vehicle}')
    print(f'  Class dist: {dict(stats)}')
    return kept

print('Building dataset...')
n_train = build_split(TRAIN_IMGS, TRAIN_LBLS, 'train', SUBSET_N)
n_val   = build_split(VAL_IMGS,   VAL_LBLS,   'val',   VAL_N)
print(f'\nTotal: {n_train} train | {n_val} val')

Building dataset...
train: 20000 images kept | no_label=0 | no_vehicle=164
  Class dist: {'car': 205514, 'truck': 3350, 'bus': 8630, 'motorcycle': 840}
val: 2000 images kept | no_label=0 | no_vehicle=22
  Class dist: {'car': 20518, 'bus': 860, 'truck': 324, 'motorcycle': 66}

Total: 20000 train | 2000 val


In [5]:
# ── Cell 5: Write data.yaml ───────────────────────────────────────────────────
import yaml
from pathlib import Path

DATASET = Path('/kaggle/working/dataset_argus')
cfg = {
    'path' : str(DATASET.resolve()),
    'train': 'train/images',
    'val'  : 'val/images',
    'nc'   : 4,
    'names': ['car', 'motorcycle', 'bus', 'truck'],
}
yaml_path = DATASET / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print(yaml_path.read_text())
print(f'Train: {sum(1 for _ in (DATASET/"train/images").iterdir())} images')
print(f'Val  : {sum(1 for _ in (DATASET/"val/images").iterdir())} images')

path: /kaggle/working/dataset_argus
train: train/images
val: val/images
nc: 4
names:
- car
- motorcycle
- bus
- truck

Train: 20000 images
Val  : 2000 images


In [6]:
# ── Cell 6: Train YOLOv8l ─────────────────────────────────────────────────────
from ultralytics import YOLO
from pathlib import Path

# yolov8l.pt downloads ~87MB COCO pretrained weights (needs internet ON)
model = YOLO('yolov8l.pt')

results = model.train(
    data          = str(Path('/kaggle/working/dataset_argus/data.yaml')),
    epochs        = 30,
    imgsz         = 640,
    batch         = 32,        # P100 16GB @ imgsz=640: safe
    device        = '0,1',         # P100 single GPU
    workers       = 2,         # conservative for Kaggle
    project       = '/kaggle/working/runs',
    name          = 'argus_bdd100k',
    exist_ok      = True,
    patience      = 10,
    lr0           = 1e-3,
    lrf           = 0.01,
    warmup_epochs = 2,
    mosaic        = 1.0,
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    fliplr        = 0.5,
    degrees       = 5.0,
    translate     = 0.1,
    scale         = 0.5,
    cache         = False,     # BDD100K too large to cache on P100
    amp           = True,      # FP16 mixed precision — P100 supports it
    close_mosaic  = 5,
    verbose       = True,
)

print(f'\nTraining complete. Best weights: {results.save_dir}/weights/best.pt')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.36 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
                                                       CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset_argus/data.yaml, degrees=5.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscrip

AttributeError: 'NoneType' object has no attribute 'save_dir'

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
import os
from pathlib import Path
from ultralytics import YOLO

best_pt = Path('/kaggle/working/runs/argus_bdd100k/weights/best.pt')
if not best_pt.exists():
    candidates = sorted(Path('/kaggle/working/runs').rglob('best.pt'), key=os.path.getmtime)
    assert candidates, 'No best.pt found — did training complete?'
    best_pt = candidates[-1]

print(f'Evaluating: {best_pt}')
model   = YOLO(str(best_pt))
metrics = model.val(
    data   = str(Path('/kaggle/working/dataset_argus/data.yaml')),
    imgsz  = 640,
    device = 0,
    verbose= True,
)

print('\n── Results ─────────────────────────────────────────')
print(f'mAP50     : {metrics.box.map50:.4f}')
print(f'mAP50-95  : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')

try:
    names = ['car', 'motorcycle', 'bus', 'truck']
    print('\nPer-class mAP50:')
    for name, ap in zip(names, metrics.box.ap50):
        bar = '█' * int(ap * 20)
        print(f'  {name:12s}: {ap:.4f}  {bar}')
except Exception as e:
    print(f'Per-class metrics unavailable: {e}')

overall = metrics.box.map50
if overall >= 0.85:
    print(f'\n TARGET EXCEEDED: {overall:.1%}')
elif overall >= 0.80:
    print(f'\n TARGET MET: {overall:.1%}')
else:
    print(f'\n Below target ({overall:.1%}) — Notebook 2 fine-tune will push it higher')

In [ ]:
# ── Cell 8: Save weights for Notebook 2 ───────────────────────────────────────
import shutil
import os
from pathlib import Path

# Find best weights
candidates = sorted(Path('/kaggle/working/runs').rglob('best.pt'), key=os.path.getmtime)
assert candidates, 'No weights found'
best_pt = candidates[-1]
last_pt = best_pt.parent / 'last.pt'

out_best = Path('/kaggle/working/argus_nb1_best.pt')
out_last = Path('/kaggle/working/argus_nb1_last.pt')

shutil.copy(best_pt, out_best)
print(f'Saved: {out_best.name}  ({out_best.stat().st_size / 1e6:.1f} MB)')

if last_pt.exists():
    shutil.copy(last_pt, out_last)
    print(f'Saved: {out_last.name}  ({out_last.stat().st_size / 1e6:.1f} MB)')

print('\n── Next Steps ───────────────────────────────────────')
print('1. Download argus_nb1_best.pt from the Output tab')
print('2. Go to kaggle.com/datasets → New Dataset → upload argus_nb1_best.pt')
print('   Name it: argus-nb1-weights')
print('3. Open Notebook 2 → add argus-nb1-weights as dataset input')
print('4. Add Talahs annotated frames as second dataset input')
print('5. Run Notebook 2')